# Task 1.2 – Sentiment Analysis with a HuggingFace Pre-trained Transformer

Fine-tunes `distilbert-base-uncased` on the Amazon cells-labelled dataset.

- `distilbert-base-uncased` via `AutoModelForSequenceClassification`
- Custom PyTorch `Dataset` + `DataLoader` — **no** `Trainer` API
- Standard TensorBoard scalars: `Loss/train`, `Loss/val`, `Accuracy/val`, `LR`, `GradNorm`
- Early stopping · label smoothing · AdamW · CosineAnnealingLR
- Model + tokenizer saved via `save_pretrained()` to `.model/25k_hf_transformer_sentiment/`

In [7]:
import os
import sys
import time
import datetime
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
import nltk


sys.path.insert(0, os.path.join(os.getcwd(), "utils"))  # jupyter path fix
from utils.preprocessing import load_data, clean_text, RANDOM_SEED, TEST_SIZE, VAL_SIZE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device {device}")

Device cuda


In [ ]:
"""Wraps raw sentence strings + integer labels and tokenises on-the-fly
with a HuggingFace tokenizer inside `__getitem__`."""


class HFSentimentDataset(Dataset):
    """
    Tokenises raw sentences on-the-fly with any HuggingFace tokenizer.
    Returns input_ids, attention_mask, and labels tensors compatible
    with AutoModelForSequenceClassification.
    """

    def __init__(self, sentences, labels, tokenizer, max_len=128):
        self.sentences = sentences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.sentences[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
DATAFILE = "data/amazon_cells_labelled_LARGE_25K.txt"
BATCH_SIZE = 32
MAX_LEN_HF = 128  # HF subword token cap
MODEL_NAME = "distilbert-base-uncased"

#  1. Run shared pipeline (saves/reuses vocab, identical splits)
vocab_path = os.path.join("model", "25k_vocab.pkl")

train_loader_shared, val_loader_shared, test_loader_shared, vocab = load_data(
    filepath=DATAFILE,
    batch_size=BATCH_SIZE,
    vocab_path=vocab_path,
)

#  2. Replay identical split on raw strings for HF tokenizer
df = pd.read_csv(DATAFILE, delimiter="\t", header=None, names=["Sentence", "Class"])
df.dropna(inplace=True)
df["Sentence"] = df["Sentence"].apply(clean_text)
df = df[df["Sentence"].str.strip() != ""].reset_index(drop=True)

sentences = df["Sentence"].tolist()
labels = df["Class"].astype(int).tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    sentences,
    labels,
    test_size=TEST_SIZE + VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=labels,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=y_temp,
)

#  3. HF tokenizer + DataLoaders
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = HFSentimentDataset(X_train, y_train, tokenizer, max_len=MAX_LEN_HF)
val_ds = HFSentimentDataset(X_val, y_val, tokenizer, max_len=MAX_LEN_HF)
test_ds = HFSentimentDataset(X_test, y_test, tokenizer, max_len=MAX_LEN_HF)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches (HF): {len(train_loader)}")
print(f"Val   batches (HF): {len(val_loader)}")
print(f"Test  batches (HF): {len(test_loader)}")

Train: 20000  Val: 2500  Test: 2500


Train batches (HF): 625
Val   batches (HF): 79
Test  batches (HF): 79


In [ ]:
def evaluate(model, loader, device):
    """Returns (avg_loss, accuracy, preds_list, labels_list)."""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            y = batch["labels"].to(device)
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            ).logits
            total_loss += criterion(logits, y).item()
            all_preds.extend(logits.argmax(-1).cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    return (
        total_loss / len(loader),
        accuracy_score(all_labels, all_preds),
        all_preds,
        all_labels,
    )

evaluate() defined.


In [ ]:
def train_hf(
    model_name=MODEL_NAME,
    n_epochs=10,
    lr=2e-5,
    weight_decay=1e-2,
    label_smoothing=0.1,
    early_stopping_patience=3,
    model_dir="model",
    log_dir="runs/25k_hf_transformer",
):
    """
    Fine-tunes a HuggingFace sequence-classification model.
    """
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, "25k_hf_transformer_sentiment")

    run_name = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    writer = SummaryWriter(log_dir=os.path.join(log_dir, run_name))
    print(f"[hf-transformer] Checkpoint   : {model_path}")
    print(f"Launch with: tensorboard --logdir {log_dir}")

    #  Model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[hf-transformer] Backbone     : {model_name}")
    print(f"[hf-transformer] Trainable params: {n_params:,}")

    #  Optimiser & scheduler
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=lr * 0.01)

    best_val_acc = 0.0
    best_state = None
    patience_ctr = 0
    ep_width = len(str(n_epochs))

    #  Training loop
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            y = batch["labels"].to(device)

            optimizer.zero_grad()
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            ).logits
            loss = criterion(logits, y)
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        train_loss = epoch_loss / len(train_loader)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, device)
        current_lr = scheduler.get_last_lr()[0]
        elapsed = time.time() - t0

        #  Standard TensorBoard scalar logging
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Accuracy/val", val_acc, epoch)
        writer.add_scalar("LR", current_lr, epoch)
        writer.add_scalar("GradNorm", float(grad_norm), epoch)

        marker = " ★" if val_acc > best_val_acc else ""
        print(
            f"Epoch {epoch:{ep_width}d}/{n_epochs}  "
            f"train_loss={train_loss:.4f}  "
            f"val_loss={val_loss:.4f}  "
            f"val_acc={val_acc:.4f}  "
            f"({elapsed:.1f}s){marker}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= early_stopping_patience:
                print(
                    f"Early stop: no val_acc improvement for {early_stopping_patience} epochs."
                )
                break

    #  Final test evaluation
    print("\n--- Test Set Evaluation (best checkpoint) ---")
    model.load_state_dict(best_state)
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, device)
    print(f"Test Loss    : {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(
        classification_report(
            test_labels,
            test_preds,
            target_names=["Negative (0)", "Positive (1)"],
        )
    )

    #  HParams summary
    writer.add_hparams(
        {
            "model_name": model_name,
            "lr": lr,
            "weight_decay": weight_decay,
            "label_smoothing": label_smoothing,
            "batch_size": BATCH_SIZE,
            "max_len": MAX_LEN_HF,
        },
        {
            "hparam/test_acc": test_acc,
            "hparam/best_val_acc": best_val_acc,
            "hparam/test_loss": test_loss,
        },
    )
    writer.close()

    #  Save model + tokenizer
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"\n[hf-transformer] Saved -> {model_path}")

    return model

In [ ]:
model = train_hf(
    model_name=MODEL_NAME,
    n_epochs=10,
    lr=2e-5,
    weight_decay=1e-2,
    label_smoothing=0.1,
    early_stopping_patience=3,
    model_dir="model",
    log_dir="runs/25k_hf_transformer",
)

[hf-transformer] Device       : cuda
[hf-transformer] Checkpoint   : model/25k_hf_transformer_sentiment
[hf-transformer] TensorBoard  : runs/25k_hf_transformer/20260418_195053
Launch with: tensorboard --logdir runs/25k_hf_transformer


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8637.01it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[hf-transformer] Backbone     : distilbert-base-uncased
[hf-transformer] Trainable params: 66,955,010
Epoch  1/10  train_loss=0.3728  val_loss=0.2266  val_acc=0.9144  (154.2s) ★
Epoch  2/10  train_loss=0.2971  val_loss=0.2211  val_acc=0.9216  (153.0s) ★
Epoch  3/10  train_loss=0.2604  val_loss=0.2353  val_acc=0.9240  (156.2s) ★
Epoch  4/10  train_loss=0.2379  val_loss=0.2491  val_acc=0.9220  (157.4s)
Epoch  5/10  train_loss=0.2253  val_loss=0.2483  val_acc=0.9228  (159.5s)


In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir runs/25k_hf_transformer

## Inference Demo

Pass arbitrary sentences through the fine-tuned model.

In [ ]:
# def predict(texts, model, tokenizer, max_len=MAX_LEN_HF):
#     """Return 'Positive' / 'Negative' for a list of raw strings."""
#     model.eval()
#     enc = tokenizer(
#         texts,
#         max_length=max_len,
#         padding="max_length",
#         truncation=True,
#         return_tensors="pt",
#     )
#     with torch.no_grad():
#         logits = model(
#             input_ids=enc["input_ids"].to(device),
#             attention_mask=enc["attention_mask"].to(device),
#         ).logits
#     return [
#         "Positive" if p == 1 else "Negative" for p in logits.argmax(-1).cpu().tolist()
#     ]


# demo_texts = [
#     "The battery life on this phone is absolutely amazing.",
#     "Worst product I have ever bought. Complete waste of money.",
#     "Decent quality for the price, nothing special.",
#     "Screen cracked on day one. Very disappointed.",
#     "Works perfectly, fast delivery, would buy again!",
# ]

# for text, label in zip(demo_texts, predict(demo_texts, model, tokenizer)):
#     print(f"[{label:8s}]  {text}")

## Reload Saved Model

Reload the fine-tuned model and tokenizer from `.model/` for a
fresh session or downstream evaluation.

In [ ]:
# reload_path = os.path.join('model', "25k_hf_transformer_sentiment")
# tokenizer_reloaded = AutoTokenizer.from_pretrained(reload_path)
# model_reloaded = AutoModelForSequenceClassification.from_pretrained(reload_path).to(
#     device
# )

# print("Model reloaded from disk.")
# _, reloaded_acc, _, _ = evaluate(model_reloaded, test_loader, device)
# print(f"Reloaded model test accuracy: {reloaded_acc:.4f}")